In [ ]:
import numpy as np
import pandas as pd

# 1. Load and inspect the box office dataset
df = pd.read_csv("15_cinema_sales.csv")
print("Shape:", df.shape)
print(df.info())
print(df.head())
print(df.describe(include="all"))


Shape: (710, 10)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 710 entries, 0 to 709
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Transaction_ID        710 non-null    object 
 1   Show_Date             710 non-null    object 
 2   Movie_Title           710 non-null    object 
 3   Genre                 710 non-null    object 
 4   Showtime_Slot         710 non-null    object 
 5   Screen_Format         710 non-null    object 
 6   Tickets_Quantity      710 non-null    int64  
 7   Ticket_Price          710 non-null    float64
 8   Occupancy_Percentage  710 non-null    float64
 9   Concession_Spend      693 non-null    float64
dtypes: float64(3), int64(1), object(6)
memory usage: 55.6+ KB
None
  Transaction_ID   Show_Date       Movie_Title      Genre Showtime_Slot  \
0         C00000  2026-10-13  The Last Journey  Animation    Late Night   
1         C00001  2025-07-05       Hidden City  

In [ ]:
# 2. Identify missing values and duplicate Transaction_ID records
print("Missing values:\n", df.isnull().sum())
dup_txn = df[df.duplicated(subset=["Transaction_ID"], keep=False)].sort_values("Transaction_ID")
print("Rows with duplicate Transaction_ID:", len(dup_txn))
print("Exact duplicate rows:", df.duplicated().sum())


Missing values:
 Transaction_ID           0
Show_Date                0
Movie_Title              0
Genre                    0
Showtime_Slot            0
Screen_Format            0
Tickets_Quantity         0
Ticket_Price             0
Occupancy_Percentage     0
Concession_Spend        17
dtype: int64
Rows with duplicate Transaction_ID: 20
Exact duplicate rows: 10


In [ ]:
# task 3: clean missing concession_spend values using a justified strategy

# Check missing values in Concession_Spend
print("Missing values before cleaning:")
print(df['Concession_Spend'].isnull().sum())

# Replace missing values with median
df['Concession_Spend'] = df['Concession_Spend'].fillna(df['Concession_Spend'].median())

# Check missing values after cleaning
print("\nMissing values after cleaning:")
print(df['Concession_Spend'].isnull().sum())

Missing values before cleaning:
17

Missing values after cleaning:
0


In [ ]:
# task 4: convert Show_Date to datetime and extract weekday vs. weekend indicators

# Convert Show_Date into datetime format
df['Show_Date'] = pd.to_datetime(df['Show_Date'])

# Extract weekday and weekend
df['Day'] = df['Show_Date'].dt.day_name()

df['Day_Type'] = np.where(df['Day'].isin(['Saturday', 'Sunday']), 'Weekend', 'Weekday')

# Display the data
print(df[['Show_Date', 'Day', 'Day_Type']].head())

   Show_Date       Day Day_Type
0 2026-10-13   Tuesday  Weekday
1 2025-07-05  Saturday  Weekend
2 2025-05-02    Friday  Weekday
3 2025-08-05   Tuesday  Weekday
4 2025-04-27    Sunday  Weekend


In [ ]:
# task 5:calculate total ticket revenue,total concession revenue,and combined revenue

# Calculate total ticket revenue
df['Ticket_Revenue'] = df['Tickets_Quantity'] * df['Ticket_Price']

# Calculate total concession revenue
df['Total_Concession_Revenue'] = df['Concession_Spend']

# Calculate combined revenue
df['Combined_Revenue'] = df['Ticket_Revenue'] + df['Total_Concession_Revenue']

# Display the results
print(df[['Movie_Title', 'Ticket_Revenue', 'Total_Concession_Revenue', 'Combined_Revenue']].head())

        Movie_Title  Ticket_Revenue  Total_Concession_Revenue  \
0  The Last Journey          616.84                    557.44   
1       Hidden City         1092.66                    355.05   
2    Laugh Out Loud          575.28                    544.21   
3  The Last Journey          432.39                    452.22   
4      Ocean Planet         1347.78                    355.05   

   Combined_Revenue  
0           1174.28  
1           1447.71  
2           1119.49  
3            884.61  
4           1702.83  


In [ ]:
# task 5:calculate total ticket revenue,total concession revenue,and combined revenue

# Calculate total ticket revenue
df['Ticket_Revenue'] = df['Tickets_Quantity'] * df['Ticket_Price']

# Calculate total concession revenue
df['Total_Concession_Revenue'] = df['Concession_Spend']

# Calculate combined revenue
df['Combined_Revenue'] = df['Ticket_Revenue'] + df['Total_Concession_Revenue']

# Display the results
print(df[['Movie_Title', 'Ticket_Revenue', 'Total_Concession_Revenue', 'Combined_Revenue']].head())

        Movie_Title  Ticket_Revenue  Total_Concession_Revenue  \
0  The Last Journey          616.84                    557.44   
1       Hidden City         1092.66                    355.05   
2    Laugh Out Loud          575.28                    544.21   
3  The Last Journey          432.39                    452.22   
4      Ocean Planet         1347.78                    355.05   

   Combined_Revenue  
0           1174.28  
1           1447.71  
2           1119.49  
3            884.61  
4           1702.83  


In [ ]:
# task 8: Compare ticket sales and concession spending on weekdays versus weekends

# Compare ticket sales and concession spending on weekdays and weekends
comparison = df.groupby('Day_Type')[['Tickets_Quantity', 'Concession_Spend']].sum()

print("Weekday vs Weekend comparison:")
print(comparison)

Weekday vs Weekend comparison:
          Tickets_Quantity  Concession_Spend
Day_Type                                    
Weekday               2087         180600.76
Weekend                772          69221.48


In [ ]:
#task 9: Analyze attendance across showtime slots (Matinee, Evening, Late Night)

# Analyze attendance across showtime slots
attendance = df.groupby('Showtime_Slot')['Tickets_Quantity'].sum().sort_values(ascending=False)

print("Attendance across showtime slots:")
print(attendance)

Attendance across showtime slots:
Showtime_Slot
Late Night    978
Matinee       963
Evening       918
Name: Tickets_Quantity, dtype: int64


In [ ]:
#task 9: Analyze attendance across showtime slots (Matinee, Evening, Late Night)

# Analyze attendance across showtime slots
attendance = df.groupby('Showtime_Slot')['Tickets_Quantity'].sum().sort_values(ascending=False)

print("Attendance across showtime slots:")
print(attendance)

Attendance across showtime slots:
Showtime_Slot
Late Night    978
Matinee       963
Evening       918
Name: Tickets_Quantity, dtype: int64


In [ ]:
# task 10: Calculate concession attach rate

# Calculate concession attach rate
total_transactions = len(df)

concession_transactions = (df['Concession_Spend'] > 0).sum()

attach_rate = (concession_transactions / total_transactions) * 100

print("Concession attach rate:", attach_rate, "%")

Concession attach rate: 100.0 %


In [ ]:
#task 11: Compare average ticket and snack spending by movie genre

# Compare average ticket and snack spending by movie genre
genre_spending = df.groupby('Genre')[['Ticket_Price', 'Concession_Spend']].mean()

print("Average spending by movie genre:")
print(genre_spending)

Average spending by movie genre:
           Ticket_Price  Concession_Spend
Genre                                    
Action       303.971085        332.912403
Animation    297.423813        363.514173
Comedy       308.064933        353.795467
Drama        315.826923        342.756573
Sci-Fi       314.781409        364.191678


In [ ]:
# task 12:Use NumPy to calculate mean, median and standard deviation of total transaction spend

# Calculate total transaction spend
df['Total_Spend'] = df['Ticket_Revenue'] + df['Concession_Spend']

# Calculate mean, median and standard deviation using NumPy
mean = np.mean(df['Total_Spend'])
median = np.median(df['Total_Spend'])
std = np.std(df['Total_Spend'])

print("Mean:", mean)
print("Median:", median)
print("Standard Deviation:", std)

Mean: 1593.6876619718314
Median: 1508.4499999999998
Standard Deviation: 773.455471409028


In [ ]:
# task 13:Identify unusually large group bookings using IQR on ticket quantity

# Find unusually large group bookings using IQR
Q1 = df['Tickets_Quantity'].quantile(0.25)
Q3 = df['Tickets_Quantity'].quantile(0.75)

IQR = Q3 - Q1

upper_limit = Q3 + 1.5 * IQR

large_bookings = df[df['Tickets_Quantity'] > upper_limit]

print("Unusually large group bookings:")
print(large_bookings[['Movie_Title', 'Tickets_Quantity']])

Unusually large group bookings:
Empty DataFrame
Columns: [Movie_Title, Tickets_Quantity]
Index: []


In [ ]:
# task 14: Create ticket-quantity bands (Solo, Couple, Small Group, Large Group)

# Create ticket quantity groups
df['Booking_Group'] = ' '

df.loc[df['Tickets_Quantity'] == 1, 'Booking_Group'] = 'Solo'
df.loc[df['Tickets_Quantity'] == 2, 'Booking_Group'] = 'Couple'
df.loc[df['Tickets_Quantity'].between(3, 5), 'Booking_Group'] = 'Small Group'
df.loc[df['Tickets_Quantity'] > 5, 'Booking_Group'] = 'Large Group'

# Display the result
print(df[['Tickets_Quantity', 'Booking_Group']].head(10))

   Tickets_Quantity Booking_Group
0                 4   Small Group
1                 6   Large Group
2                 2        Couple
3                 1          Solo
4                 6   Large Group
5                 1          Solo
6                 7   Large Group
7                 1          Solo
8                 5   Small Group
9                 4   Small Group


In [ ]:
# task 15:Write five marketing and scheduling recommendations
# This task is generally written in words after looking at your results. You can write recommendations like these:
# 1. Movies with higher ticket sales should be given more screen slots to increase revenue.
# 2. Weekend shows should be scheduled according to the higher attendance observed on weekends.
# 3. Combo offers should be introduced to encourage customers to purchase snacks along with tickets.
# 4. Group discounts can be offered to attract small and large groups of customers.
# 5. Movies with lower attendance can be scheduled during less popular time slots to improve screen utilization